In [1]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

# Buscar el archivo
archivo = "noticias_preparadas.parquet"
rutas = [Path(archivo), Path("/content") / archivo, Path("/content/drive/MyDrive") / archivo]
ruta_valida = next((r for r in rutas if r.exists()), None)

if not ruta_valida:
    print(f"❌ No se encontró '{archivo}'. Súbelo a Colab o monta tu Drive.")
else:
    print(f"✅ Archivo encontrado en: {ruta_valida}")

    # 1. Leer esquema y detectar columnas
    df = pd.read_parquet(ruta_valida)
    cols = df.columns.tolist()

    col_id = "id" if "id" in cols else None
    col_texto = next((c for c in ["texto_limpio", "texto", "text"] if c in cols), None)

    if not col_id or not col_texto:
        print(f"❌ Faltan columnas vitales. Columnas actuales: {cols}")
    else:
        print(f"✅ Columna ID detectada: '{col_id}' | Columna Texto detectada: '{col_texto}'")

        # 2. Limpieza básica para el modelo (quitar nulos o textos vacíos)
        total_previo = len(df)
        df[col_texto] = df[col_texto].astype(str).str.strip()
        df = df[(df[col_texto].notna()) & (df[col_texto] != "")]

        print(f"🧹 Limpieza: Se descartaron {total_previo - len(df)} filas sin texto válido.")
        print(f"📊 Total de noticias listas para clasificar: {len(df)}")

        # Guardar una copia temporal segura para el modelo
        df.to_parquet("/content/noticias_listas_para_lsthf.parquet", index=False)
        print("✅ Archivo temporal creado: '/content/noticias_listas_para_lsthf.parquet'. ¡Listo para el modelo!")

✅ Archivo encontrado en: noticias_preparadas.parquet
✅ Columna ID detectada: 'id' | Columna Texto detectada: 'texto_limpio'
🧹 Limpieza: Se descartaron 0 filas sin texto válido.
📊 Total de noticias listas para clasificar: 78360
✅ Archivo temporal creado: '/content/noticias_listas_para_lsthf.parquet'. ¡Listo para el modelo!


In [3]:
import gc
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import re

# ==============================================================================
# 1. CONFIGURACIÓN DEL PIPELINE
# ==============================================================================
CONFIG = {
    "archivo_entrada": "/content/noticias_listas_para_lsthf.parquet",
    "dir_resultados": Path("/content/resultados_lsthf"),
    "modelo_id": "Lsthf/multilingual-emotion-classification",
    "batch_size": 32,              # Reducido a 32 porque XLM-R es más pesado
    "max_tokens": 512,             # Capacidad nativa de XLM-RoBERTa
    "umbral_deteccion": 0.5,       # Umbral estándar para clasificación sigmoide
    "exportar_csv": True
}

CONFIG["dir_resultados"].mkdir(parents=True, exist_ok=True)
RE_ORACION = re.compile(r"(?<=[.!?…»”\"])\s+")

# ==============================================================================
# 2. CARGA DE MODELO Y TOKENIZADOR
# ==============================================================================
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if dispositivo == "cuda" else torch.float32

print(f"Cargando {CONFIG['modelo_id']} en {dispositivo.upper()}...")
tok = AutoTokenizer.from_pretrained(CONFIG["modelo_id"])
modelo = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["modelo_id"], torch_dtype=dtype
).to(dispositivo).eval()

etiquetas = [modelo.config.id2label[i] for i in range(len(modelo.config.id2label))]
n_etq = len(etiquetas)
print(f"✅ Modelo cargado. {n_etq} Emociones a detectar: {', '.join(etiquetas)}")

# Extraer tokens especiales (ej. <s> y </s> para XLM-R)
con_esp = tok("a", add_special_tokens=True)["input_ids"]
sin_esp = tok("a", add_special_tokens=False)["input_ids"]
pre, suf = [], []
for i in range(len(con_esp) - len(sin_esp) + 1):
    if con_esp[i:i+len(sin_esp)] == sin_esp:
        pre, suf = con_esp[:i], con_esp[i+len(sin_esp):]
        break

presupuesto = CONFIG["max_tokens"] - len(pre) - len(suf)
pad_id = tok.pad_token_id or 0

# ==============================================================================
# 3. FUNCIONES DE PROCESAMIENTO
# ==============================================================================
def tokenizar_y_fragmentar(texto, tok, presupuesto):
    oraciones = [s for s in RE_ORACION.split(texto) if s.strip()]

    # Tokenizamos indicando explícitamente que no trunque, nosotros lo haremos
    ids_or = tok(oraciones, add_special_tokens=False, truncation=False)["input_ids"]

    frags = []
    cur_ids = []
    for ids in ids_or:
        if not ids:
            continue

        # Si la "oración" gigante supera el límite (ej. 968 tokens), la partimos en trozos
        if len(ids) > presupuesto:
            if cur_ids:
                frags.append(cur_ids)
                cur_ids = []

            # Sub-fragmentación exacta sin perder texto
            for i in range(0, len(ids), presupuesto):
                frags.append(ids[i : i + presupuesto])
            continue

        if len(cur_ids) + len(ids) > presupuesto:
            frags.append(cur_ids)
            cur_ids = []
        cur_ids.extend(ids)

    if cur_ids:
        frags.append(cur_ids)
    return frags

def predecir_lote(lista_ids):
    n = len(lista_ids)
    out = np.zeros((n, n_etq), dtype=np.float32)
    secuencias = [pre + ids + suf for ids in lista_ids]
    max_len = max(len(s) for s in secuencias)

    ids_t = torch.full((n, max_len), pad_id, dtype=torch.long)
    mask = torch.zeros((n, max_len), dtype=torch.long)

    for r, s in enumerate(secuencias):
        ids_t[r, :len(s)] = torch.tensor(s, dtype=torch.long)
        mask[r, :len(s)] = 1

    with torch.inference_mode():
        logits = modelo(input_ids=ids_t.to(dispositivo), attention_mask=mask.to(dispositivo)).logits

    return torch.sigmoid(logits.float()).cpu().numpy()

# ==============================================================================
# 4. EJECUCIÓN PRINCIPAL
# ==============================================================================
df = pd.read_parquet(CONFIG["archivo_entrada"])
col_texto = next(c for c in ["texto_limpio", "texto", "text"] if c in df.columns)

resultados = []
lote_ids, lote_meta = [], []

barra = tqdm(total=len(df), desc="Procesando noticias")

# Procesar fila por fila y agrupar en lotes por fragmento
for _, fila in df.iterrows():
    texto = fila[col_texto]
    if not texto:
        barra.update(1)
        continue

    frags = tokenizar_y_fragmentar(texto, tok, presupuesto)

    for f_idx, f_ids in enumerate(frags):
        lote_ids.append(f_ids)
        lote_meta.append({
            "id": fila["id"],
            "idioma": fila.get("idioma", "unknown"),
            "n_tokens": len(f_ids),
            "frag_idx": f_idx,
            "total_frags": len(frags)
        })

        if len(lote_ids) >= CONFIG["batch_size"]:
            probs = predecir_lote(lote_ids)
            for m, pr in zip(lote_meta, probs):
                m.update({f"p_{e}": pr[i] for i, e in enumerate(etiquetas)})
                resultados.append(m)
            lote_ids, lote_meta = [], []

    barra.update(1)

# Procesar el último lote sobrante
if lote_ids:
    probs = predecir_lote(lote_ids)
    for m, pr in zip(lote_meta, probs):
        m.update({f"p_{e}": pr[i] for i, e in enumerate(etiquetas)})
        resultados.append(m)

barra.close()

# ==============================================================================
# 5. AGREGACIÓN Y GUARDADO
# ==============================================================================
df_res = pd.DataFrame(resultados)

# Agrupamos los fragmentos para sacar el promedio por noticia (ponderado por longitud)
def agregar_noticia(g):
    w = g["n_tokens"].values
    sw = w.sum()
    if sw == 0: return pd.Series({f"media_{e}": 0.0 for e in etiquetas})

    res = {}
    for e in etiquetas:
        res[f"media_{e}"] = np.average(g[f"p_{e}"].values, weights=w)
        res[f"det_{e}"] = res[f"media_{e}"] >= CONFIG["umbral_deteccion"]

    res["emociones_detectadas"] = ";".join([e for e in etiquetas if res[f"det_{e}"]])
    return pd.Series(res)

print("Agregando resultados a nivel de noticia...")
noticias_final = df_res.groupby(["id", "idioma"]).apply(agregar_noticia).reset_index()

noticias_final.to_parquet(CONFIG["dir_resultados"] / "lsthf_emociones_noticias.parquet")
if CONFIG["exportar_csv"]:
    noticias_final.to_csv(CONFIG["dir_resultados"] / "lsthf_emociones_noticias.csv", index=False)

print(f"✅ ¡Proceso terminado! Archivos guardados en {CONFIG['dir_resultados']}")

Cargando Lsthf/multilingual-emotion-classification en CUDA...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Modelo cargado. 11 Emociones a detectar: anger, contempt, disgust, fear, frustration, gratitude, joy, love, neutral, sadness, surprise


Procesando noticias:   0%|          | 0/78360 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (968 > 512). Running this sequence through the model will result in indexing errors


Agregando resultados a nivel de noticia...


/tmp/ipykernel_614/1713495854.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  noticias_final = df_res.groupby(["id", "idioma"]).apply(agregar_noticia).reset_index()


✅ ¡Proceso terminado! Archivos guardados en /content/resultados_lsthf
